# 01 — Technical data audit

This notebook performs a bounded technical audit of all immutable CSV files in `data/raw/`. It covers schemas, types, completeness, candidate keys, relationships, basic distributions, journey coverage, and pre-defined quality checks. It does **not** select a business problem, define product metrics, or perform open-ended EDA.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from ecommerce_product_analytics.data_audit import (
    build_journey_coverage,
    build_quality_findings,
    load_tables,
    profile_columns,
    profile_keys,
    profile_relationships,
    profile_tables,
    value_distribution,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'raw').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

tables = load_tables(RAW_DATA_DIR)
print(f'Loaded {len(tables)} tables and {sum(len(table) for table in tables.values()):,} rows.')

Loaded 9 tables and 1,550,922 rows.


## Table overview

Exact duplicates are counted across complete rows. Missing cells are shown as a technical completeness measure, not as a judgment that every null is erroneous.

In [2]:
table_profile = profile_tables(tables)
display(table_profile)

,table,rows,columns,exact_duplicate_rows,missing_cells
0,category_translation,71,2,0,0
1,customers,99441,5,0,0
2,geolocation,1000163,5,261831,0
3,order_items,112650,7,0,0
4,order_payments,103886,5,0,0
5,order_reviews,99224,7,0,145903
6,orders,99441,8,0,4908
7,products,32951,9,0,2448
8,sellers,3095,4,0,0


## Column profile

Known timestamps are parsed strictly. Postal prefixes are loaded as strings so leading zeroes are preserved. Numeric and timestamp ranges are descriptive only.

In [3]:
column_profile = profile_columns(tables)
with pd.option_context('display.max_rows', 100, 'display.max_columns', 20):
    display(column_profile)

,table,column,dtype,missing_count,missing_pct,unique_non_null,minimum,median,maximum
0,customers,customer_id,object,0,0.000000,99441,None,None,None
1,customers,customer_unique_id,object,0,0.000000,96096,None,None,None
2,customers,customer_zip_code_prefix,string,0,0.000000,14994,None,None,None
3,customers,customer_city,object,0,0.000000,4119,None,None,None
4,customers,customer_state,object,0,0.000000,27,None,None,None
5,geolocation,geolocation_zip_code_prefix,string,0,0.000000,19015,None,None,None
6,geolocation,geolocation_lat,float64,0,0.000000,717363,-36.605374,-22.919377,45.065933
7,geolocation,geolocation_lng,float64,0,0.000000,717615,-101.466766,-46.637879,121.105394
8,geolocation,geolocation_city,object,0,0.000000,8011,None,None,None
9,geolocation,geolocation_state,object,0,0.000000,27,None,None,None


## Candidate keys and relationships

A candidate key must be both complete and unique. Relationships marked `strict=True` connect core business entities; lookup coverage for category translation and geolocation is reported separately.

In [4]:
key_profile = profile_keys(tables)
relationship_profile = profile_relationships(tables)
display(key_profile)
display(relationship_profile)

,table,key,columns,null_key_rows,duplicate_key_rows,is_candidate_key
0,customers,customers.customer_id,customer_id,0,0,True
1,order_items,"order_items.(order_id, order_item_id)","order_id, order_item_id",0,0,True
2,order_payments,"order_payments.(order_id, payment_sequential)","order_id, payment_sequential",0,0,True
3,order_reviews,"order_reviews.(review_id, order_id)","review_id, order_id",0,0,True
4,orders,orders.order_id,order_id,0,0,True
5,products,products.product_id,product_id,0,0,True
6,sellers,sellers.seller_id,seller_id,0,0,True
7,category_translation,category_translation.product_category_name,product_category_name,0,0,True


,relationship,strict,child_rows,child_null_rows,orphan_rows,orphan_distinct_values
0,orders.customer_id → customers.customer_id,True,99441,0,0,0
1,order_items.order_id → orders.order_id,True,112650,0,0,0
2,order_payments.order_id → orders.order_id,True,103886,0,0,0
3,order_reviews.order_id → orders.order_id,True,99224,0,0,0
4,order_items.product_id → products.product_id,True,112650,0,0,0
5,order_items.seller_id → sellers.seller_id,True,112650,0,0,0
6,products.category → category_translation.category,False,32951,610,13,2
7,customers.zip_prefix → geolocation.zip_prefix,False,99441,0,278,157
8,sellers.zip_prefix → geolocation.zip_prefix,False,3095,0,7,7


## Bounded categorical distributions

Only three central low-cardinality fields are shown. This is intentionally not an open-ended EDA.

In [5]:
for table_name, column in [
    ('orders', 'order_status'),
    ('order_payments', 'payment_type'),
    ('order_reviews', 'review_score'),
]:
    print(f'{table_name}.{column}')
    display(value_distribution(tables[table_name], column))

orders.order_status


,value,count,pct
0,delivered,96478,0.970203
1,shipped,1107,0.011132
2,canceled,625,0.006285
3,unavailable,609,0.006124
4,invoiced,314,0.003158
5,processing,301,0.003027
6,created,5,0.000050
7,approved,2,0.000020


order_payments.payment_type


,value,count,pct
0,credit_card,76795,0.739224
1,boleto,19784,0.190440
2,voucher,5775,0.055590
3,debit_card,1529,0.014718
4,not_defined,3,0.000029


order_reviews.review_score


,value,count,pct
0,5,57328,0.577763
1,4,19142,0.192917
2,1,11424,0.115133
3,3,8179,0.082430
4,2,3151,0.031756


## Journey coverage and bounded quality findings

Counts below describe what can be linked and where source anomalies require handling. A non-zero count may be a valid business state, missing context, or a data defect; the audit does not silently recode any value.

In [6]:
journey_coverage = build_journey_coverage(tables)
quality_findings = build_quality_findings(tables)
display(journey_coverage)
with pd.option_context('display.max_colwidth', 100):
    display(quality_findings)

,metric,value
0,orders,99441
1,orders_with_items,98666
2,orders_with_payments,99440
3,orders_with_reviews,98673
4,unique_customer_identifiers,96096
5,repeat_customer_identifiers,2997
6,orders_with_multiple_items,9803
7,orders_with_multiple_payment_rows,2961
8,orders_with_multiple_review_rows,547


,check,count,note
0,geolocation_exact_duplicate_rows,261831,Exact duplicates; geolocation is not a unique ZIP dimension.
1,review_id_duplicate_rows,814,review_id alone is not unique.
2,review_order_id_duplicate_rows,551,Some orders have more than one review row.
3,products_missing_catalog_metadata_rows,610,"Category, name length, description length, and photo count all missing."
4,products_missing_physical_metadata_rows,2,All weight and dimension fields missing.
5,products_zero_weight_rows,4,Zero weight is suspicious but not overwritten.
6,products_without_category_translation_rows,13,Two Portuguese category values have no English lookup.
7,customer_rows_without_geolocation_zip,278,Postal lookup coverage gap; not a strict entity foreign key.
8,seller_rows_without_geolocation_zip,7,Postal lookup coverage gap; not a strict entity foreign key.
9,zero_value_payment_rows,9,Includes vouchers and undefined payment types.


## Audit boundary

The data supports reconstruction of the observed post-purchase order lifecycle and repeat orders within the observation window. It does not contain visits, impressions, searches, carts, acquisition channels, inventory history, costs, experiment assignment, or reliable source metadata. Product investigation directions must therefore be chosen only after reviewing these coverage limits.